In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT=Path.cwd().parents[1]
sys.path.insert(0,str(ROOT))

import common as cfg
import helper_funcs as hf

CONFIG=cfg.load_config()
INPUT,OUTPUT,SRC=cfg.task_paths()
OUTPUT.mkdir(parents=True,exist_ok=True)
rng=cfg.initialize(CONFIG['seed'])


In [ ]:
targets=pd.read_csv(INPUT/'b_prepare'/'empirical_targets.csv')
followup=pd.read_csv(INPUT/'b_prepare'/'followup_distribution.csv')


In [ ]:
results=[]
for index,model in enumerate(CONFIG['models']):
    model_rng=np.random.default_rng(cfg.model_seed(CONFIG['seed'],model))
    fitted=hf.abc_smc(model,CONFIG,followup,targets,model_rng)
    results.append(fitted)
    print(model,'final epsilon',fitted.loc[fitted['generation']==fitted['generation'].max(),'epsilon_actual'].iloc[0])
particles=pd.concat(results,ignore_index=True)


In [ ]:
final=particles.loc[particles['generation']==particles.groupby('model')['generation'].transform('max')]
diagnostics=final.groupby('model',as_index=False).agg(
    particles=('particle','size'),
    median_distance=('distance','median'),
    best_distance=('distance','min'),
    epsilon=('epsilon_actual','first'),
    relaxed=('relaxed','max'),
    weight_ess=('weight',lambda x:1/np.sum(np.asarray(x)**2))
).sort_values('median_distance')
particles.to_csv(OUTPUT/'abc_particles.csv',index=False)
diagnostics.to_csv(OUTPUT/'abc_diagnostics.csv',index=False)


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4.5))
epsilon=particles.groupby(['model','generation'],as_index=False)['epsilon_actual'].first()
sns.lineplot(data=epsilon,x='generation',y='epsilon_actual',hue='model',style='model',markers=True,dashes=True,ax=axes[0])
sns.barplot(data=diagnostics,y='model',x='median_distance',ax=axes[1])
axes[0].set(xlabel='ABC generation',ylabel='accepted tolerance')
axes[1].set(xlabel='final median distance',ylabel='')
fig.tight_layout()
fig.savefig(OUTPUT/'abc_diagnostics.png',dpi=220)
plt.close(fig)
diagnostics
